In [1]:
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langchain.messages import HumanMessage

from dotenv import load_dotenv
load_dotenv(override=True)

# 连接模型
model = ChatDeepSeek(
    model='deepseek-v4-flash',
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)
# 预定义状态 继承 MessagesState 不是Typeddict
class OverAllState(MessagesState):
    username: str
    output: str

def node_a(state: OverAllState) -> OverAllState:
    return {
        "messages": [HumanMessage("你好，我是" + state["username"])]
    }

def llm_node(state: OverAllState) -> OverAllState:
    res = model.invoke(state["messages"])

    return {
        "messages": [res],
        "output": res.content
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()
response = graph.invoke({"username": "老吴"})
print(response)

{'messages': [HumanMessage(content='你好，我是 老吴', additional_kwargs={}, response_metadata={}, id='06feb60d-9909-470a-b2c0-2d03f7783db5'), AIMessage(content='你好，老吴！👋\n\n很高兴认识你！我是DeepSeek，你的AI助手。无论你有什么问题、想聊聊天，还是需要帮忙处理什么事情，我都在这里随时待命。\n\n尽管开口吧——无论是工作上的难题、学习中的困惑，还是想聊聊生活趣事，或者需要一些建议，我都会认真倾听，尽我所能帮助你！😊\n\n你今天想聊点什么呢？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 10, 'total_tokens': 95, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 10}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': 'f0fe05b3-1e31-4d2d-9456-dc68ad3862e2', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a03c82-d95a-7990-b27b-914b38240fa9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 85, 'total_tokens': 95, 'input

`OverAllState` 继承了 `MessagesState`，因此可用的状态字段为

```
messages
username
output
```

